In [1]:
import json
import os
from typing import Optional, Dict, Tuple

def extract_client_and_adapter_idx(path: str, default_client: Optional[int] = None) -> Tuple[Optional[int], Optional[int]]:
    """
    파일에서 '첫 번째 유효 JSON 라인'을 찾아 파싱한 뒤,
    (client, adapter_idx)를 반환합니다.
    """
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue  # 첫 줄이 깨졌으면 다음 줄도 시도(원하면 break로 바꿔도 됨)
            return obj.get("client", default_client), obj.get("adapter_idx")
    return default_client, None


def collect_adapter_idx_by_client(root='.', start=1, end=53, prefix='client_', ext='.raw') -> Dict[int, int]:
    """
    각 client 파일에서 adapter_idx만 추출해서 {client_idx: adapter_idx} 로 반환합니다.
    - client_idx 키는 JSON의 "client" 값을 우선 사용하고, 없으면 파일 인덱스(i)를 사용합니다.
    - 파일이 없거나 adapter_idx가 없으면 스킵합니다.
    """
    out: Dict[int, int] = {}
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            client, adapter_idx = extract_client_and_adapter_idx(path, default_client=i)
        except FileNotFoundError:
            continue

        if client is None or adapter_idx is None:
            continue
        out[int(client)] = int(adapter_idx)  # 중복이면 최신이 덮어씀

    return out



d = collect_adapter_idx_by_client(root=".", start=1, end=40, prefix="client_", ext=".raw")
d_plus1 = {k: v + 1 for k, v in d.items()}
print(d_plus1)

{1: 5, 2: 5, 3: 5, 4: 5, 5: 5, 6: 5, 7: 5, 8: 5, 9: 5, 10: 5, 11: 7, 12: 7, 13: 7, 14: 7, 15: 7, 16: 7, 17: 7, 18: 7, 19: 7, 20: 7, 21: 6, 22: 6, 23: 6, 24: 6, 25: 1, 26: 6, 27: 1, 28: 1, 29: 6, 30: 6, 31: 4, 32: 2, 33: 2, 34: 4, 35: 2, 36: 2, 37: 2, 38: 2, 39: 2, 40: 2}
